<a href="https://colab.research.google.com/github/jgte29/ucbhonorsthesis2023-2024/blob/main/SF_DA_Data_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# !pip install firebase-admin

# Load in Data & Libraries

In [3]:
import firebase_admin
from firebase_admin import credentials
from firebase_admin import db
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import re
from scipy import stats
import warnings
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.diagnostic import linear_lm
import sklearn
from sklearn.model_selection import train_test_split
from datetime import datetime
# Initialize the Firebase app with your service account key
cred = credentials.Certificate(r'PRIVATE-KEY-sf-da-analysis-firebase.json')
firebase_admin.initialize_app(cred, {
    'databaseURL': 'https://sf-da-analysis-6f156-default-rtdb.firebaseio.com/'
})

In [4]:
# For sf_prosecutions
# Get a reference to the Firebase Realtime Database
ref = db.reference('/District_Attorney_Cases_Prosecuted_2014_2023')

# Load data from the collection
sf_prosecutions = ref.get()

# Process the loaded data & converts it to a df
sf_prosecutions = pd.json_normalize(sf_prosecutions)
sf_prosecutions = sf_prosecutions.drop('id', axis=1)

# For sf_case_resolutions_all
# Get a reference to the Firebase Realtime Database
ref = db.reference('/Case_Resoultions_All_Crimes_2015_2023')

# Load data from the collection
sf_case_resolutions_all = ref.get()

# Process the loaded data & converts it to a df
sf_case_resolutions_all = pd.json_normalize(sf_case_resolutions_all)
sf_case_resolutions_all = sf_case_resolutions_all.drop('id', axis=1)

# For sf_case_resolutions_violent
# Get a reference to the Firebase Realtime Database
ref = db.reference('/Case_Resoultions_Violent_Crimes_2015_2023')

# Load data from the collection
sf_case_resolutions_violent = ref.get()

# Process the loaded data & converts it to a df
sf_case_resolutions_violent = pd.json_normalize(sf_case_resolutions_violent)
sf_case_resolutions_violent = sf_case_resolutions_violent.drop('id', axis=1)

# For sf_da_actions_all
# Get a reference to the Firebase Realtime Database
ref = db.reference('/DA_Actions_On_Arrests_All_Crimes_2015_2023')

# Load data from the collection
sf_da_actions_all = ref.get()

# Process the loaded data & converts it to a df
sf_da_actions_all = pd.json_normalize(sf_da_actions_all)
sf_da_actions_all = sf_da_actions_all.drop('id', axis=1)

# For sf_da_actions_violent
# Get a reference to the Firebase Realtime Database
ref = db.reference('/DA_Actions_On_Arrests_Violent_Crimes_2015_2023')

# Load data from the collection
sf_da_actions_violent = ref.get()

# Process the loaded data & converts it to a df
sf_da_actions_violent = pd.json_normalize(sf_da_actions_violent)
sf_da_actions_violent = sf_da_actions_violent.drop('id', axis=1)

# For sfpd_incidents_outcomes_all
# Get a reference to the Firebase Realtime Database
ref = db.reference('/Outcomes_of_SFPD_Incidents_All_Crimes_2015_2023')

# Load data from the collection
sfpd_incidents_outcomes_all = ref.get()

# Process the loaded data & converts it to a df
sfpd_incidents_outcomes_all = pd.json_normalize(sfpd_incidents_outcomes_all)
sfpd_incidents_outcomes_all = sfpd_incidents_outcomes_all.drop('id', axis=1)

# For fed_off_by_crime_all
# Get a reference to the Firebase Realtime Database
ref = db.reference('/Federal_Offenders_by_Type_of_Crime_All_Crimes_2015_2022')

# Load data from the collection
fed_off_by_crime_all = ref.get()

# Process the loaded data & converts it to a df
fed_off_by_crime_all = pd.json_normalize(fed_off_by_crime_all)
fed_off_by_crime_all = fed_off_by_crime_all.drop('id', axis=1)

# For fed_off_by_crime_violent
# Get a reference to the Firebase Realtime Database
ref = db.reference('/Federal_Offenders_by_Type_of_Crime_Violent_Crimes_2015_2022')

# Load data from the collection
fed_off_by_crime_violent = ref.get()

# Process the loaded data & converts it to a df
fed_off_by_crime_violent = pd.json_normalize(fed_off_by_crime_violent)
fed_off_by_crime_violent = fed_off_by_crime_violent.drop('id', axis=1)

# For fed_off_by_dist_all
# Get a reference to the Firebase Realtime Database
ref = db.reference('/Federal_Offenders_in_Each_District_All_Crimes_2015_2022')

# Load data from the collection
fed_off_by_dist_all = ref.get()

# Process the loaded data & converts it to a df
fed_off_by_dist_all = pd.json_normalize(fed_off_by_dist_all)
fed_off_by_dist_all = fed_off_by_dist_all.drop('id', axis=1)

# For fed_off_by_dist_violent
# Get a reference to the Firebase Realtime Database
ref = db.reference('/Federal_Offenders_in_Each_District_Violent_Crimes_2015_2022')

# Load data from the collection
fed_off_by_dist_violent = ref.get()

# Process the loaded data & converts it to a df
fed_off_by_dist_violent = pd.json_normalize(fed_off_by_dist_violent)
fed_off_by_dist_violent = fed_off_by_dist_violent.drop('id', axis=1)

In [5]:
fed_off_by_dist_all

,Circuit,District,Fiscal_Year,N,Pct
0,Grand Total,Total,2022,64157,1.000
1,DC Circuit,District of Columbia,2022,312,0.005
2,1st Circuit,Maine,2022,180,0.003
3,1st Circuit,Massachusetts,2022,466,0.007
4,1st Circuit,New Hampshire,2022,180,0.003
...,...,...,...,...,...
755,11th Circuit,"Florida, Northern",2015,255,0.004
756,11th Circuit,"Florida, Southern",2015,2238,0.032
757,11th Circuit,"Georgia, Middle",2015,377,0.005
758,11th Circuit,"Georgia, Northern",2015,592,0.008


In [6]:
date_format = "%Y-%m-%dT%H:%M:%S.%fZ"

sf_prosecutions['arrest_date'] = pd.to_datetime(sf_prosecutions['arrest_date'], format = date_format)
sf_prosecutions['data_as_of'] = pd.to_datetime(sf_prosecutions['data_as_of'], format = date_format)
sf_prosecutions['data_loaded_at'] = pd.to_datetime(sf_prosecutions['data_loaded_at'], format = date_format)

In [7]:
boudin_start_date = datetime(2020, 1, 8, 0, 0, 0)
boudin_end_date = datetime(2022, 7, 9, 0, 0, 0)
sf_prosecutions_boudin = sf_prosecutions[(sf_prosecutions["arrest_date"] >= boudin_start_date) &
                                         (sf_prosecutions["arrest_date"] < boudin_end_date)]

sf_prosecutions_boudin

,arrest_date,case_status,court_number,crime_type,da_action_taken,data_as_of,data_loaded_at,dv_case,filed_case_type,incident_number,list_of_filed_charges
9383,2020-01-09 08:00:00,Filed Motion to Revoke,2506152,"Forgery, Checks, Access Cards",New charges filed,2023-08-28 07:00:00,2023-08-31 05:01:04,No,MTR,Z20171107-02506152,470D/F/0
9384,2020-01-30 08:00:00,Filed Motion to Revoke,2520522,Theft,New charges filed,2023-08-28 07:00:00,2023-08-31 05:01:04,No,MTR,Z20200115-02520522,"487C/F/0, 245A4/M/0"
9385,2020-01-14 08:00:00,Filed Motion to Revoke,14005420,Assault,New charges filed,2023-08-28 07:00:00,2023-08-31 05:01:04,Yes,MTR,140153084,"591.5/M/0, 273.5A/F/0, 273.5A/F/0, 273.5A/F/0,..."
9386,2020-01-22 08:00:00,Filed Motion to Revoke,15018930,Robbery,New charges filed,2023-08-28 07:00:00,2023-08-31 05:01:04,No,MTR,150735638,"242/M/0, 245A4/F/0, 243B/M/0, 211/F/2"
9387,2020-01-16 08:00:00,Filed Motion to Revoke,15026261,Narcotics,New charges filed,2023-08-28 07:00:00,2023-08-31 05:01:04,No,MTR,150025953,"11378/F/0, 11377/M/0, 23123A/I/0"
...,...,...,...,...,...,...,...,...,...,...,...
99090,2022-06-24 07:00:00,,2530936,,New charges filed,2023-08-28 07:00:00,2023-08-31 05:01:04,No,,23-00344,
99091,2022-06-13 07:00:00,Pending,2530799,,New charges filed,2023-08-28 07:00:00,2023-08-31 05:01:04,No,,Z20220613-02530799,
99092,2022-07-06 07:00:00,Fugitive Status,22006648,,New charges filed,2023-08-28 07:00:00,2023-08-31 05:01:04,No,Felony,220445185,"664/1085A/F/0, 664/496DA/F/0"
99096,2022-07-04 07:00:00,Pled Guilty to Other Case or Other DA Action,22006556,,New charges filed,2023-08-28 07:00:00,2023-08-31 05:01:04,No,,220438726,


# PART I - Boudin v National Trends (DID)

## All Crimes

In [ ]:
def year_marker(year, start, end=9999, mode = 'range'):
  # Default of 9999 is used to indicate ongoing
  if mode == 'before':
    try:
      if year < start:
        return 1
      else:
        return 0
    except:
      return 0

  if mode == 'after':
    start += 1 #Increments start by 1

  # Used for both 'after' and 'range' modes
  else:
    try:
      if year >= start and year <= end:
        return 1
      else:
        return 0
    except:
      return 0

In [ ]:
fed_off_totals_all = fed_off_by_dist_all[fed_off_by_dist_all["District"] == "Total"].reset_index(drop=True)
fed_off_totals_all = fed_off_totals_all.iloc[0:, 2:5].drop("Pct", axis = 1)

#Renames the column to be consistent with future dfs
fed_off_totals_all.rename(columns={'Fiscal_Year': 'Year'}, inplace=True)
fed_off_totals_all.rename(columns={'N': 'Total'}, inplace=True)

fed_off_totals_all['Log_Total'] = np.log(fed_off_totals_all['Total'])
fed_off_totals_all['Boudin'] = 0
fed_off_totals_all['SF'] = 0
fed_off_totals_all['Before_COVID'] = fed_off_totals_all.apply(lambda row :
                                                              year_marker(row['Year'],
                                                                          2020, mode = 'before'),
                                                              axis = 1)
fed_off_totals_all

,Year,Total,Log_Total,Boudin,SF,Before_COVID
0,2022,64157,11.069088,0,0,0
1,2021,57304,10.956126,0,0,0
2,2020,64591,11.075830,0,0,0
3,2019,76593,11.246261,0,0,1
4,2018,69470,11.148650,0,0,1
5,2017,66899,11.110939,0,0,1
6,2016,67794,11.124229,0,0,1
7,2015,71030,11.170858,0,0,1


In [ ]:
# This temporarily disables warnings; A warning of a depreciated opperation displayed
warnings.filterwarnings('ignore')

sfda_convictions = sf_case_resolutions_all[["Year", "Conviction"]]
sfda_convictions["Conviction"] = sfda_convictions["Conviction"].astype(int)

#Renames the column to be consistent with future dfs
sfda_convictions.rename(columns={'Conviction': 'Total'}, inplace=True)

sfda_convictions['Log_Total'] = np.log(sfda_convictions['Total'])
sfda_convictions['Boudin'] = sfda_convictions.apply(lambda row : year_marker(row['Year'],
                                                                             2020, 2022),
                                                    axis = 1)
sfda_convictions['SF'] = 1
sfda_convictions['Before_COVID'] = sfda_convictions.apply(lambda row :
                                                              year_marker(row['Year'],
                                                                          2020, mode = 'before'),
                                                              axis = 1)

sfda_convictions_proj = sfda_convictions.copy() #Creates defensive copy
sfda_convictions_proj.iloc[8, 1] = sfda_convictions_proj.iloc[9, 1]

# Drops '2023*' Row
sfda_convictions_proj = sfda_convictions_proj.drop(9)

# Drops '2023*' Row
sfda_convictions = sfda_convictions.drop(9)

sfda_convictions["Year"] = sfda_convictions["Year"].astype(int)
sfda_convictions_proj["Year"] = sfda_convictions_proj["Year"].astype(int)

# This resets warnings
warnings.resetwarnings()

In [ ]:
# Concatenate vertically (along rows)
sf_fed_off_totals = pd.concat([fed_off_totals_all, sfda_convictions], axis=0)
sf_fed_off_totals_proj = pd.concat([fed_off_totals_all, sfda_convictions_proj], axis=0)

# Reset the index
sf_fed_off_totals = sf_fed_off_totals.reset_index(drop=True)
sf_fed_off_totals_proj = sf_fed_off_totals_proj.reset_index(drop=True)

sf_fed_off_totals

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


,Year,Total,Log_Total,Boudin,SF,Before_COVID
0,2022,64157,11.069088,0,0,0
1,2021,57304,10.956126,0,0,0
2,2020,64591,11.075830,0,0,0
3,2019,76593,11.246261,0,0,1
4,2018,69470,11.148650,0,0,1
5,2017,66899,11.110939,0,0,1
6,2016,67794,11.124229,0,0,1
7,2015,71030,11.170858,0,0,1
8,2015,3460,8.149024,0,1,1
9,2016,3482,8.155362,0,1,1


### 2023 Inclusive

In [ ]:
#2023 Raw & Total NOT Log Normalized
np.random.seed(29)
formula = "Total ~ Boudin + Year + SF + Before_COVID + Boudin*Year"
model = smf.gls(formula=formula, data=sf_fed_off_totals)
result = model.fit()
print(result.summary())

                            GLS Regression Results                            
Dep. Variable:                  Total   R-squared:                       0.994
Model:                            GLS   Adj. R-squared:                  0.992
Method:                 Least Squares   F-statistic:                     395.2
Date:                Wed, 20 Sep 2023   Prob (F-statistic):           5.06e-12
Time:                        23:55:35   Log-Likelihood:                -156.57
No. Observations:                  17   AIC:                             325.1
Df Residuals:                      11   BIC:                             330.1
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept    -1.588e+06   1.23e+06     -1.289   

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)
/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=17
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [ ]:
#2023 Raw & Total Log Normalized
np.random.seed(29)
formula = "Log_Total ~ Boudin + Year + SF + Before_COVID + Boudin*Year"
model = smf.gls(formula=formula, data=sf_fed_off_totals)
result = model.fit()
print(result.summary())

                            GLS Regression Results                            
Dep. Variable:              Log_Total   R-squared:                       0.991
Model:                            GLS   Adj. R-squared:                  0.987
Method:                 Least Squares   F-statistic:                     244.3
Date:                Wed, 20 Sep 2023   Prob (F-statistic):           6.96e-11
Time:                        23:55:35   Log-Likelihood:                 6.8828
No. Observations:                  17   AIC:                            -1.766
Df Residuals:                      11   BIC:                             3.234
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept       67.9599     82.269      0.826   

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)
/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=17
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [ ]:
#2023 Projected & Total NOT Log Normalized
np.random.seed(29)
formula = "Total ~ Boudin + Year + SF + Before_COVID + Boudin*Year"
model = smf.gls(formula=formula, data=sf_fed_off_totals_proj)
result = model.fit()
print(result.summary())

                            GLS Regression Results                            
Dep. Variable:                  Total   R-squared:                       0.994
Model:                            GLS   Adj. R-squared:                  0.992
Method:                 Least Squares   F-statistic:                     381.5
Date:                Wed, 20 Sep 2023   Prob (F-statistic):           6.14e-12
Time:                        23:55:35   Log-Likelihood:                -156.85
No. Observations:                  17   AIC:                             325.7
Df Residuals:                      11   BIC:                             330.7
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept    -1.644e+06   1.25e+06     -1.312   

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)
/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=17
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [ ]:
#2023 Projected & Total Log Normalized
np.random.seed(29)
formula = "Log_Total ~ Boudin + Year + SF + Before_COVID + Boudin*Year"
model = smf.gls(formula=formula, data=sf_fed_off_totals_proj)
result = model.fit()
print(result.summary())

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)
/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=17
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


                            GLS Regression Results                            
Dep. Variable:              Log_Total   R-squared:                       0.991
Model:                            GLS   Adj. R-squared:                  0.987
Method:                 Least Squares   F-statistic:                     244.3
Date:                Wed, 20 Sep 2023   Prob (F-statistic):           6.96e-11
Time:                        23:55:35   Log-Likelihood:                 6.8828
No. Observations:                  17   AIC:                            -1.766
Df Residuals:                      11   BIC:                             3.234
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept       67.9599     82.269      0.826   

### 2023 Exclusive

In [ ]:
# Drops '2023' Row
sf_fed_off_totals_no2023 = sf_fed_off_totals[sf_fed_off_totals["Year"] != 2023]
sf_fed_off_totals_no2023

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


,Year,Total,Log_Total,Boudin,SF,Before_COVID
0,2022,64157,11.069088,0,0,0
1,2021,57304,10.956126,0,0,0
2,2020,64591,11.075830,0,0,0
3,2019,76593,11.246261,0,0,1
4,2018,69470,11.148650,0,0,1
5,2017,66899,11.110939,0,0,1
6,2016,67794,11.124229,0,0,1
7,2015,71030,11.170858,0,0,1
8,2015,3460,8.149024,0,1,1
9,2016,3482,8.155362,0,1,1


In [ ]:
# Total NOT Log Normalized
np.random.seed(29)
formula = "Total ~ Boudin + Year + SF + Before_COVID + Boudin*Year"
model = smf.gls(formula=formula, data=sf_fed_off_totals_no2023)
result = model.fit()
print(result.summary())

                            GLS Regression Results                            
Dep. Variable:                  Total   R-squared:                       0.995
Model:                            GLS   Adj. R-squared:                  0.992
Method:                 Least Squares   F-statistic:                     393.9
Date:                Wed, 20 Sep 2023   Prob (F-statistic):           3.84e-11
Time:                        23:55:35   Log-Likelihood:                -146.62
No. Observations:                  16   AIC:                             305.2
Df Residuals:                      10   BIC:                             309.9
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept    -1.092e+06   1.26e+06     -0.867   

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)
/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=16
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [ ]:
# Total Log Normalized
np.random.seed(29)
formula = "Log_Total ~ Boudin + Year + SF + Before_COVID + Boudin*Year"
model = smf.gls(formula=formula, data=sf_fed_off_totals_no2023)
result = model.fit()
print(result.summary())

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)
/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=16
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


                            GLS Regression Results                            
Dep. Variable:              Log_Total   R-squared:                       0.998
Model:                            GLS   Adj. R-squared:                  0.997
Method:                 Least Squares   F-statistic:                     858.8
Date:                Wed, 20 Sep 2023   Prob (F-statistic):           7.92e-13
Time:                        23:55:35   Log-Likelihood:                 17.576
No. Observations:                  16   AIC:                            -23.15
Df Residuals:                      10   BIC:                            -18.52
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept       -9.1745     43.966     -0.209   

## Violent Crimes

In [ ]:
fed_off_totals_violent = fed_off_by_dist_violent[fed_off_by_dist_violent["District"] == "Total"].reset_index(drop=True)
fed_off_totals_violent = fed_off_totals_violent.iloc[0:, 2:5].drop("Pct", axis = 1)

#Renames the column to be consistent with future dfs
fed_off_totals_violent.rename(columns={'Fiscal_Year': 'Year'}, inplace=True)
fed_off_totals_violent.rename(columns={'N': 'Total'}, inplace=True)

fed_off_totals_violent['Log_Total'] = np.log(fed_off_totals_violent['Total'])
fed_off_totals_violent['Boudin'] = 0
fed_off_totals_violent['SF'] = 0
fed_off_totals_violent['Before_COVID'] = fed_off_totals_violent.apply(lambda row :
                                                              year_marker(row['Year'],
                                                                          2020, mode = 'before'),
                                                              axis = 1)
fed_off_totals_violent

,Year,Total,Log_Total,Boudin,SF,Before_COVID
0,2022,4417,8.393216,0,0,0
1,2021,3438,8.142645,0,0,0
2,2020,3219,8.076826,0,0,0
3,2019,4326,8.372399,0,0,1
4,2018,3973,8.287277,0,0,1
5,2017,3927,8.275631,0,0,1
6,2016,3816,8.246958,0,0,1
7,2015,3835,8.251925,0,0,1


In [ ]:
# This temporarily disables warnings; A warning of a depreciated opperation displayed
warnings.filterwarnings('ignore')

sfda_convictions_violent = sf_case_resolutions_violent[["Year", "Conviction"]]
sfda_convictions_violent["Conviction"] = sfda_convictions_violent["Conviction"].astype(int)

#Renames the column to be consistent with future dfs
sfda_convictions_violent.rename(columns={'Conviction': 'Total'}, inplace=True)

sfda_convictions_violent['Log_Total'] = np.log(sfda_convictions_violent['Total'])
sfda_convictions_violent['Boudin'] = sfda_convictions_violent.apply(lambda row : year_marker(row['Year'],
                                                                             2020, 2022),
                                                    axis = 1)
sfda_convictions_violent['SF'] = 1
sfda_convictions_violent['Before_COVID'] = sfda_convictions_violent.apply(lambda row :
                                                              year_marker(row['Year'],
                                                                          2020, mode = 'before'),
                                                              axis = 1)

sfda_convictions_violent_proj = sfda_convictions_violent.copy() #Creates defensive copy
sfda_convictions_violent_proj.iloc[8, 1] = sfda_convictions_violent_proj.iloc[9, 1]

# Drops '2023*' Row
sfda_convictions_violent_proj = sfda_convictions_violent_proj.drop(9)

# Drops '2023*' Row
sfda_convictions_violent = sfda_convictions_violent.drop(9)

sfda_convictions_violent["Year"] = sfda_convictions_violent["Year"].astype(int)
sfda_convictions_violent_proj["Year"] = sfda_convictions_violent_proj["Year"].astype(int)

# This resets warnings
warnings.resetwarnings()

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [ ]:
# Concatenate vertically (along rows)
sf_fed_off_totals_violent = pd.concat([fed_off_totals_violent, sfda_convictions_violent], axis=0)
sf_fed_off_totals_violent_proj = pd.concat([fed_off_totals_violent, sfda_convictions_violent_proj], axis=0)

# Reset the index
sf_fed_off_totals_violent = sf_fed_off_totals_violent.reset_index(drop=True)
sf_fed_off_totals_violent_proj = sf_fed_off_totals_violent_proj.reset_index(drop=True)

sf_fed_off_totals_violent

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


,Year,Total,Log_Total,Boudin,SF,Before_COVID
0,2022,4417,8.393216,0,0,0
1,2021,3438,8.142645,0,0,0
2,2020,3219,8.076826,0,0,0
3,2019,4326,8.372399,0,0,1
4,2018,3973,8.287277,0,0,1
5,2017,3927,8.275631,0,0,1
6,2016,3816,8.246958,0,0,1
7,2015,3835,8.251925,0,0,1
8,2015,1039,6.946014,0,1,1
9,2016,972,6.879356,0,1,1


### 2023 Inclusive

In [ ]:
#2023 Raw & Total NOT Log Normalized
np.random.seed(29)
formula = "Total ~ Boudin + Year + SF + Before_COVID + Boudin*Year"
model = smf.gls(formula=formula, data=sf_fed_off_totals_violent)
result = model.fit()
print(result.summary())

                            GLS Regression Results                            
Dep. Variable:                  Total   R-squared:                       0.979
Model:                            GLS   Adj. R-squared:                  0.969
Method:                 Least Squares   F-statistic:                     101.8
Date:                Thu, 21 Sep 2023   Prob (F-statistic):           7.89e-09
Time:                        01:18:56   Log-Likelihood:                -116.56
No. Observations:                  17   AIC:                             245.1
Df Residuals:                      11   BIC:                             250.1
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept    -1.711e+05   1.17e+05     -1.460   

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)
/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=17
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [ ]:
#2023 Raw & Total Log Normalized
np.random.seed(29)
formula = "Log_Total ~ Boudin + Year + SF + Before_COVID + Boudin*Year"
model = smf.gls(formula=formula, data=sf_fed_off_totals_violent)
result = model.fit()
print(result.summary())

                            GLS Regression Results                            
Dep. Variable:              Log_Total   R-squared:                       0.960
Model:                            GLS   Adj. R-squared:                  0.942
Method:                 Least Squares   F-statistic:                     52.82
Date:                Thu, 21 Sep 2023   Prob (F-statistic):           2.56e-07
Time:                        01:18:57   Log-Likelihood:                 5.3307
No. Observations:                  17   AIC:                             1.339
Df Residuals:                      11   BIC:                             6.338
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept       28.2685     90.134      0.314   

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)
/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=17
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [ ]:
#2023 Projected & Total NOT Log Normalized
np.random.seed(29)
formula = "Total ~ Boudin + Year + SF + Before_COVID + Boudin*Year"
model = smf.gls(formula=formula, data=sf_fed_off_totals_violent_proj)
result = model.fit()
print(result.summary())

                            GLS Regression Results                            
Dep. Variable:                  Total   R-squared:                       0.980
Model:                            GLS   Adj. R-squared:                  0.972
Method:                 Least Squares   F-statistic:                     110.5
Date:                Thu, 21 Sep 2023   Prob (F-statistic):           5.10e-09
Time:                        01:18:57   Log-Likelihood:                -115.76
No. Observations:                  17   AIC:                             243.5
Df Residuals:                      11   BIC:                             248.5
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept    -1.876e+05   1.12e+05     -1.679   

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)
/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=17
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [ ]:
#2023 Projected & Total Log Normalized
np.random.seed(29)
formula = "Log_Total ~ Boudin + Year + SF + Before_COVID + Boudin*Year"
model = smf.gls(formula=formula, data=sf_fed_off_totals_violent_proj)
result = model.fit()
print(result.summary())

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


                            GLS Regression Results                            
Dep. Variable:              Log_Total   R-squared:                       0.960
Model:                            GLS   Adj. R-squared:                  0.942
Method:                 Least Squares   F-statistic:                     52.82
Date:                Thu, 21 Sep 2023   Prob (F-statistic):           2.56e-07
Time:                        01:18:57   Log-Likelihood:                 5.3307
No. Observations:                  17   AIC:                             1.339
Df Residuals:                      11   BIC:                             6.338
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept       28.2685     90.134      0.314   

/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=17
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


### 2023 Exclusive

In [ ]:
# Drops '2023' Row
sf_fed_off_totals_violent_no2023 = sf_fed_off_totals_violent[sf_fed_off_totals_violent["Year"] != 2023]
sf_fed_off_totals_violent_no2023

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


,Year,Total,Log_Total,Boudin,SF,Before_COVID
0,2022,4417,8.393216,0,0,0
1,2021,3438,8.142645,0,0,0
2,2020,3219,8.076826,0,0,0
3,2019,4326,8.372399,0,0,1
4,2018,3973,8.287277,0,0,1
5,2017,3927,8.275631,0,0,1
6,2016,3816,8.246958,0,0,1
7,2015,3835,8.251925,0,0,1
8,2015,1039,6.946014,0,1,1
9,2016,972,6.879356,0,1,1


In [ ]:
# Total NOT Log Normalized
np.random.seed(29)
formula = "Total ~ Boudin + Year + SF + Before_COVID + Boudin*Year"
model = smf.gls(formula=formula, data=sf_fed_off_totals_violent_no2023)
result = model.fit()
print(result.summary())

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


                            GLS Regression Results                            
Dep. Variable:                  Total   R-squared:                       0.981
Model:                            GLS   Adj. R-squared:                  0.971
Method:                 Least Squares   F-statistic:                     101.8
Date:                Thu, 21 Sep 2023   Prob (F-statistic):           3.04e-08
Time:                        01:18:57   Log-Likelihood:                -108.74
No. Observations:                  16   AIC:                             229.5
Df Residuals:                      10   BIC:                             234.1
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept    -2.221e+05   1.18e+05     -1.883   

/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=16
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [ ]:
# Total Log Normalized
np.random.seed(29)
formula = "Log_Total ~ Boudin + Year + SF + Before_COVID + Boudin*Year"
model = smf.gls(formula=formula, data=sf_fed_off_totals_violent_no2023)
result = model.fit()
print(result.summary())

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


                            GLS Regression Results                            
Dep. Variable:              Log_Total   R-squared:                       0.990
Model:                            GLS   Adj. R-squared:                  0.985
Method:                 Least Squares   F-statistic:                     194.6
Date:                Thu, 21 Sep 2023   Prob (F-statistic):           1.26e-09
Time:                        01:18:57   Log-Likelihood:                 16.810
No. Observations:                  16   AIC:                            -21.62
Df Residuals:                      10   BIC:                            -16.98
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept      -57.3137     46.120     -1.243   

/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=16
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


# PART II - SF Before & After Boudin Takes Office (RDD)

## All Crimes

### 2023 Inclusive

In [ ]:
# This temporarily disables warnings; A warning of a depreciated opperation displayed
warnings.filterwarnings('ignore')

sf_cr_all = sf_case_resolutions_all.copy()
sf_cr_all["Conviction"] = sf_cr_all["Conviction"].astype(int)

sf_cr_all['Log_Conviction'] = np.log(sf_cr_all['Conviction'])
sf_cr_all['Log_Total'] = np.log(sf_cr_all['Total'])
sf_cr_all['Boudin'] = sf_cr_all.apply(lambda row : year_marker(row['Year'],
                                                                             2020, 2022),
                                                    axis = 1)
sf_cr_all['Before_COVID'] = sf_cr_all.apply(lambda row :
                                                              year_marker(row['Year'],
                                                                          2020, mode = 'before'),
                                                              axis = 1)



sf_cr_all_proj = sf_cr_all.copy() #Creates defensive copy
sf_cr_all_proj.iloc[8, 1] = sf_cr_all_proj.iloc[9, 1] #Moves 2023* to 2023 slot

# Drops '2023*' Row
sf_cr_all_proj = sf_cr_all_proj.drop(9)

# Drops '2023*' Row
sf_cr_all = sf_cr_all.drop(9)

# Addresses type issues
sf_cr_all["Year"] = sf_cr_all["Year"].astype(int)
sf_cr_all["Med_Arrest_To_Close"] = sf_cr_all["Med_Arrest_To_Close"].astype(int)
sf_cr_all["Med_Arrest_To_Close_Conviction"] = sf_cr_all["Med_Arrest_To_Close_Conviction"].astype(int)
sf_cr_all["Med_Arrest_To_Close_Dismissal"] = sf_cr_all["Med_Arrest_To_Close_Dismissal"].astype(int)

sf_cr_all_proj["Year"] = sf_cr_all_proj["Year"].astype(int)
sf_cr_all_proj["Med_Arrest_To_Close"] = sf_cr_all_proj["Med_Arrest_To_Close"].astype(int)
sf_cr_all_proj["Med_Arrest_To_Close_Conviction"] = sf_cr_all_proj["Med_Arrest_To_Close_Conviction"].astype(int)
sf_cr_all_proj["Med_Arrest_To_Close_Dismissal"] = sf_cr_all_proj["Med_Arrest_To_Close_Dismissal"].astype(int)

# Gets rid of 2023 Row
sf_cr_all_no2023 = sf_cr_all[sf_cr_all["Year"] != 2023]

# This resets warnings
warnings.resetwarnings()

In [ ]:
# 2023 Raw & Total Log Normalized
np.random.seed(29)
formula = "Log_Total ~ Boudin + Year + Before_COVID"
model = smf.gls(formula=formula, data=sf_cr_all)
result = model.fit()
print(result.summary())

                            GLS Regression Results                            
Dep. Variable:              Log_Total   R-squared:                       0.880
Model:                            GLS   Adj. R-squared:                  0.809
Method:                 Least Squares   F-statistic:                     12.26
Date:                Thu, 21 Sep 2023   Prob (F-statistic):            0.00965
Time:                        01:24:21   Log-Likelihood:                 11.177
No. Observations:                   9   AIC:                            -14.35
Df Residuals:                       5   BIC:                            -13.57
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept      -84.9060     54.757     -1.551   

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)
/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=9
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [ ]:
# 2023 Projected & Total Log Normalized
np.random.seed(29)
formula = "Log_Total ~ Boudin + Year + Before_COVID"
model = smf.gls(formula=formula, data=sf_cr_all_proj)
result = model.fit()
print(result.summary())

                            GLS Regression Results                            
Dep. Variable:              Log_Total   R-squared:                       0.880
Model:                            GLS   Adj. R-squared:                  0.809
Method:                 Least Squares   F-statistic:                     12.26
Date:                Thu, 21 Sep 2023   Prob (F-statistic):            0.00965
Time:                        01:24:21   Log-Likelihood:                 11.177
No. Observations:                   9   AIC:                            -14.35
Df Residuals:                       5   BIC:                            -13.57
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept      -84.9060     54.757     -1.551   

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)
/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=9
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [ ]:
# 2023 Raw & Conviction Log Normalized
np.random.seed(29)
formula = "Log_Conviction ~ Boudin + Year + Before_COVID"
model = smf.gls(formula=formula, data=sf_cr_all)
result = model.fit()
print(result.summary())

                            GLS Regression Results                            
Dep. Variable:         Log_Conviction   R-squared:                       0.916
Model:                            GLS   Adj. R-squared:                  0.866
Method:                 Least Squares   F-statistic:                     18.23
Date:                Thu, 21 Sep 2023   Prob (F-statistic):            0.00401
Time:                        01:24:22   Log-Likelihood:                 6.0050
No. Observations:                   9   AIC:                            -4.010
Df Residuals:                       5   BIC:                            -3.221
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept       56.9768     97.283      0.586   

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)
/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=9
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [ ]:
# 2023 Projected & Conviction Log Normalized
np.random.seed(29)
formula = "Log_Conviction ~ Boudin + Year + Before_COVID"
model = smf.gls(formula=formula, data=sf_cr_all_proj)
result = model.fit()
print(result.summary())

                            GLS Regression Results                            
Dep. Variable:         Log_Conviction   R-squared:                       0.916
Model:                            GLS   Adj. R-squared:                  0.866
Method:                 Least Squares   F-statistic:                     18.23
Date:                Thu, 21 Sep 2023   Prob (F-statistic):            0.00401
Time:                        01:24:22   Log-Likelihood:                 6.0050
No. Observations:                   9   AIC:                            -4.010
Df Residuals:                       5   BIC:                            -3.221
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept       56.9768     97.283      0.586   

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)
/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=9
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [ ]:
# 2023 Raw & Successful_Diversion Not Log Normalized
np.random.seed(29)
formula = "Successful_Diversion ~ Boudin + Year + Before_COVID"
model = smf.gls(formula=formula, data=sf_cr_all)
result = model.fit()
print(result.summary())

                             GLS Regression Results                             
Dep. Variable:     Successful_Diversion   R-squared:                       0.913
Model:                              GLS   Adj. R-squared:                  0.861
Method:                   Least Squares   F-statistic:                     17.52
Date:                  Thu, 21 Sep 2023   Prob (F-statistic):            0.00439
Time:                          01:24:22   Log-Likelihood:                -55.955
No. Observations:                     9   AIC:                             119.9
Df Residuals:                         5   BIC:                             120.7
Df Model:                             3                                         
Covariance Type:              nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept    -3.254e+05    9

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)
/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=9
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [ ]:
# 2023 Projected & Successful_Diversion Not Log Normalized
np.random.seed(29)
formula = "Successful_Diversion ~ Boudin + Year + Before_COVID"
model = smf.gls(formula=formula, data=sf_cr_all_proj)
result = model.fit()
print(result.summary())

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)
/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=9
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


                             GLS Regression Results                             
Dep. Variable:     Successful_Diversion   R-squared:                       0.913
Model:                              GLS   Adj. R-squared:                  0.861
Method:                   Least Squares   F-statistic:                     17.52
Date:                  Thu, 21 Sep 2023   Prob (F-statistic):            0.00439
Time:                          01:24:22   Log-Likelihood:                -55.955
No. Observations:                     9   AIC:                             119.9
Df Residuals:                         5   BIC:                             120.7
Df Model:                             3                                         
Covariance Type:              nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept    -3.254e+05    9

In [ ]:
# 2023 Raw & Med_Arrest_To_Close Not Log Normalized
np.random.seed(29)
formula = "Med_Arrest_To_Close ~ Boudin + Year + Before_COVID"
model = smf.gls(formula=formula, data=sf_cr_all)
result = model.fit()
print(result.summary())

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)
/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=9
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


                             GLS Regression Results                            
Dep. Variable:     Med_Arrest_To_Close   R-squared:                       0.963
Model:                             GLS   Adj. R-squared:                  0.941
Method:                  Least Squares   F-statistic:                     43.63
Date:                 Thu, 21 Sep 2023   Prob (F-statistic):           0.000522
Time:                         01:24:22   Log-Likelihood:                -39.339
No. Observations:                    9   AIC:                             86.68
Df Residuals:                        5   BIC:                             87.47
Df Model:                            3                                         
Covariance Type:             nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept    -4.058e+04    1.5e+04    

In [ ]:
# 2023 Projected & Med_Arrest_To_Close Not Log Normalized
np.random.seed(29)
formula = "Med_Arrest_To_Close ~ Boudin + Year + Before_COVID"
model = smf.gls(formula=formula, data=sf_cr_all_proj)
result = model.fit()
print(result.summary())

                             GLS Regression Results                            
Dep. Variable:     Med_Arrest_To_Close   R-squared:                       0.963
Model:                             GLS   Adj. R-squared:                  0.941
Method:                  Least Squares   F-statistic:                     43.63
Date:                 Thu, 21 Sep 2023   Prob (F-statistic):           0.000522
Time:                         01:24:22   Log-Likelihood:                -39.339
No. Observations:                    9   AIC:                             86.68
Df Residuals:                        5   BIC:                             87.47
Df Model:                            3                                         
Covariance Type:             nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept    -4.058e+04    1.5e+04    

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)
/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=9
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [ ]:
# 2023 Raw & Med_Arrest_To_Close_Conviction Not Log Normalized
np.random.seed(29)
formula = "Med_Arrest_To_Close_Conviction ~ Boudin + Year + Before_COVID"
model = smf.gls(formula=formula, data=sf_cr_all)
result = model.fit()
print(result.summary())

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


                                  GLS Regression Results                                  
Dep. Variable:     Med_Arrest_To_Close_Conviction   R-squared:                       0.993
Model:                                        GLS   Adj. R-squared:                  0.989
Method:                             Least Squares   F-statistic:                     242.1
Date:                            Thu, 21 Sep 2023   Prob (F-statistic):           7.85e-06
Time:                                    01:24:22   Log-Likelihood:                -28.819
No. Observations:                               9   AIC:                             65.64
Df Residuals:                                   5   BIC:                             66.43
Df Model:                                       3                                         
Covariance Type:                        nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
---------

/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=9
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [ ]:
# 2023 Projected & Med_Arrest_To_Close_Conviction Not Log Normalized
np.random.seed(29)
formula = "Med_Arrest_To_Close_Conviction ~ Boudin + Year + Before_COVID"
model = smf.gls(formula=formula, data=sf_cr_all_proj)
result = model.fit()
print(result.summary())

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


                                  GLS Regression Results                                  
Dep. Variable:     Med_Arrest_To_Close_Conviction   R-squared:                       0.993
Model:                                        GLS   Adj. R-squared:                  0.989
Method:                             Least Squares   F-statistic:                     242.1
Date:                            Thu, 21 Sep 2023   Prob (F-statistic):           7.85e-06
Time:                                    01:24:22   Log-Likelihood:                -28.819
No. Observations:                               9   AIC:                             65.64
Df Residuals:                                   5   BIC:                             66.43
Df Model:                                       3                                         
Covariance Type:                        nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
---------

/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=9
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [ ]:
# 2023 Projected & Med_Arrest_To_Close_Dismissal Not Log Normalized
np.random.seed(29)
formula = "Med_Arrest_To_Close_Dismissal ~ Boudin + Year + Before_COVID"
model = smf.gls(formula=formula, data=sf_cr_all_proj)
result = model.fit()
print(result.summary())

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)
/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=9
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


                                  GLS Regression Results                                 
Dep. Variable:     Med_Arrest_To_Close_Dismissal   R-squared:                       0.927
Model:                                       GLS   Adj. R-squared:                  0.883
Method:                            Least Squares   F-statistic:                     21.18
Date:                           Thu, 21 Sep 2023   Prob (F-statistic):            0.00285
Time:                                   01:24:22   Log-Likelihood:                -38.402
No. Observations:                              9   AIC:                             84.80
Df Residuals:                                  5   BIC:                             85.59
Df Model:                                      3                                         
Covariance Type:                       nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
-------------------

In [ ]:
# 2023 Raw & Total Log Normalized
np.random.seed(29)
formula = "Log_Total ~ Boudin + Year + Before_COVID + Med_Arrest_To_Close + Med_Arrest_To_Close_Conviction + Med_Arrest_To_Close_Dismissal"
model = smf.gls(formula=formula, data=sf_cr_all)
result = model.fit()
print(result.summary())

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


                            GLS Regression Results                            
Dep. Variable:              Log_Total   R-squared:                       0.999
Model:                            GLS   Adj. R-squared:                  0.997
Method:                 Least Squares   F-statistic:                     425.7
Date:                Thu, 21 Sep 2023   Prob (F-statistic):            0.00235
Time:                        01:24:22   Log-Likelihood:                 33.812
No. Observations:                   9   AIC:                            -53.62
Df Residuals:                       2   BIC:                            -52.24
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                                     coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
Intercept   

/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=9
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [ ]:
# 2023 Projected & Total Log Normalized
np.random.seed(29)
formula = "Log_Total ~ Boudin + Year + Before_COVID + Med_Arrest_To_Close + Med_Arrest_To_Close_Conviction + Med_Arrest_To_Close_Dismissal"
model = smf.gls(formula=formula, data=sf_cr_all_proj)
result = model.fit()
print(result.summary())

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


                            GLS Regression Results                            
Dep. Variable:              Log_Total   R-squared:                       0.999
Model:                            GLS   Adj. R-squared:                  0.997
Method:                 Least Squares   F-statistic:                     425.7
Date:                Thu, 21 Sep 2023   Prob (F-statistic):            0.00235
Time:                        01:24:22   Log-Likelihood:                 33.812
No. Observations:                   9   AIC:                            -53.62
Df Residuals:                       2   BIC:                            -52.24
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                                     coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
Intercept   

/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=9
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [ ]:
# 2023 Raw & Conviction Log Normalized
np.random.seed(29)
formula = "Log_Conviction ~ Boudin + Year + Before_COVID + Med_Arrest_To_Close + Med_Arrest_To_Close_Conviction + Med_Arrest_To_Close_Dismissal"
model = smf.gls(formula=formula, data=sf_cr_all)
result = model.fit()
print(result.summary())

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)
/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=9
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


                            GLS Regression Results                            
Dep. Variable:         Log_Conviction   R-squared:                       0.997
Model:                            GLS   Adj. R-squared:                  0.989
Method:                 Least Squares   F-statistic:                     117.4
Date:                Thu, 21 Sep 2023   Prob (F-statistic):            0.00847
Time:                        01:24:22   Log-Likelihood:                 21.248
No. Observations:                   9   AIC:                            -28.50
Df Residuals:                       2   BIC:                            -27.12
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                                     coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
Intercept   

In [ ]:
# 2023 Projected & Conviction Log Normalized
np.random.seed(29)
formula = "Log_Conviction ~ Boudin + Year + Before_COVID + Med_Arrest_To_Close + Med_Arrest_To_Close_Conviction + Med_Arrest_To_Close_Dismissal"
model = smf.gls(formula=formula, data=sf_cr_all_proj)
result = model.fit()
print(result.summary())

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)
/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=9
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


                            GLS Regression Results                            
Dep. Variable:         Log_Conviction   R-squared:                       0.997
Model:                            GLS   Adj. R-squared:                  0.989
Method:                 Least Squares   F-statistic:                     117.4
Date:                Thu, 21 Sep 2023   Prob (F-statistic):            0.00847
Time:                        01:24:22   Log-Likelihood:                 21.248
No. Observations:                   9   AIC:                            -28.50
Df Residuals:                       2   BIC:                            -27.12
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                                     coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
Intercept   

### 2023 Exclusive
(Note: Can't Use Before_COVID because between (2015 & 2022), Boudin & Before_COVID are **Linearly Dependent**)

In [ ]:
# Total Log Normalized
np.random.seed(29)
formula = "Log_Total ~ Boudin + Year"
model = smf.gls(formula=formula, data=sf_cr_all_no2023)
result = model.fit()
print(result.summary())

                            GLS Regression Results                            
Dep. Variable:              Log_Total   R-squared:                       0.829
Model:                            GLS   Adj. R-squared:                  0.760
Method:                 Least Squares   F-statistic:                     12.10
Date:                Thu, 21 Sep 2023   Prob (F-statistic):             0.0121
Time:                        01:24:27   Log-Likelihood:                 9.4643
No. Observations:                   8   AIC:                            -12.93
Df Residuals:                       5   BIC:                            -12.69
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept    -84.1583     54.595     -1.542      0.1

/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=8
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [ ]:
# Conviction Log Normalized
np.random.seed(29)
formula = "Log_Conviction ~ Boudin + Year"
model = smf.gls(formula=formula, data=sf_cr_all_no2023)
result = model.fit()
print(result.summary())

                            GLS Regression Results                            
Dep. Variable:         Log_Conviction   R-squared:                       0.895
Model:                            GLS   Adj. R-squared:                  0.853
Method:                 Least Squares   F-statistic:                     21.28
Date:                Thu, 21 Sep 2023   Prob (F-statistic):            0.00358
Time:                        01:24:28   Log-Likelihood:                 4.8666
No. Observations:                   8   AIC:                            -3.733
Df Residuals:                       5   BIC:                            -3.495
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     57.7411     96.994      0.595      0.5

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)
/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=8
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [ ]:
# Successful_Diversion Not Log Normalized
np.random.seed(29)
formula = "Successful_Diversion ~ Boudin + Year"
model = smf.gls(formula=formula, data=sf_cr_all_no2023)
result = model.fit()
print(result.summary())

                             GLS Regression Results                             
Dep. Variable:     Successful_Diversion   R-squared:                       0.911
Model:                              GLS   Adj. R-squared:                  0.876
Method:                   Least Squares   F-statistic:                     25.68
Date:                  Thu, 21 Sep 2023   Prob (F-statistic):            0.00234
Time:                          01:24:28   Log-Likelihood:                -50.209
No. Observations:                     8   AIC:                             106.4
Df Residuals:                         5   BIC:                             106.7
Df Model:                             2                                         
Covariance Type:              nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept  -3.249e+05   9.48e+04

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)
/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=8
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [ ]:
# Med_Arrest_To_Close Not Log Normalized
np.random.seed(29)
formula = "Med_Arrest_To_Close ~ Boudin + Year"
model = smf.gls(formula=formula, data=sf_cr_all_no2023)
result = model.fit()
print(result.summary())

                             GLS Regression Results                            
Dep. Variable:     Med_Arrest_To_Close   R-squared:                       0.956
Model:                             GLS   Adj. R-squared:                  0.939
Method:                  Least Squares   F-statistic:                     54.88
Date:                 Thu, 21 Sep 2023   Prob (F-statistic):           0.000396
Time:                         01:24:28   Log-Likelihood:                -35.439
No. Observations:                    8   AIC:                             76.88
Df Residuals:                        5   BIC:                             77.12
Df Model:                            2                                         
Covariance Type:             nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept  -4.065e+04    1.5e+04     -2.71

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)
/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=8
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [ ]:
# Med_Arrest_To_Close_Conviction Not Log Normalized
np.random.seed(29)
formula = "Med_Arrest_To_Close_Conviction ~ Boudin + Year"
model = smf.gls(formula=formula, data=sf_cr_all_no2023)
result = model.fit()
print(result.summary())

                                  GLS Regression Results                                  
Dep. Variable:     Med_Arrest_To_Close_Conviction   R-squared:                       0.992
Model:                                        GLS   Adj. R-squared:                  0.989
Method:                             Least Squares   F-statistic:                     305.1
Date:                            Thu, 21 Sep 2023   Prob (F-statistic):           5.95e-06
Time:                                    01:24:28   Log-Likelihood:                -26.088
No. Observations:                               8   AIC:                             58.18
Df Residuals:                                   5   BIC:                             58.41
Df Model:                                       2                                         
Covariance Type:                        nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
-----------

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)
/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=8
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [ ]:
# 2023 Projected & Med_Arrest_To_Close_Dismissal Not Log Normalized
np.random.seed(29)
formula = "Med_Arrest_To_Close_Dismissal ~ Boudin + Year"
model = smf.gls(formula=formula, data=sf_cr_all_proj)
result = model.fit()
print(result.summary())

                                  GLS Regression Results                                 
Dep. Variable:     Med_Arrest_To_Close_Dismissal   R-squared:                       0.807
Model:                                       GLS   Adj. R-squared:                  0.742
Method:                            Least Squares   F-statistic:                     12.51
Date:                           Thu, 21 Sep 2023   Prob (F-statistic):            0.00724
Time:                                   01:24:28   Log-Likelihood:                -42.790
No. Observations:                              9   AIC:                             91.58
Df Residuals:                                  6   BIC:                             92.17
Df Model:                                      2                                         
Covariance Type:                       nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
---------------------

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)
/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=9
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [ ]:
# Total Log Normalized
np.random.seed(29)
formula = "Log_Total ~ Boudin + Year + Med_Arrest_To_Close + Med_Arrest_To_Close_Conviction + Med_Arrest_To_Close_Dismissal"
model = smf.gls(formula=formula, data=sf_cr_all)
result = model.fit()
print(result.summary())

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)
/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=9
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


                            GLS Regression Results                            
Dep. Variable:              Log_Total   R-squared:                       0.990
Model:                            GLS   Adj. R-squared:                  0.973
Method:                 Least Squares   F-statistic:                     59.24
Date:                Thu, 21 Sep 2023   Prob (F-statistic):            0.00338
Time:                        01:24:28   Log-Likelihood:                 22.334
No. Observations:                   9   AIC:                            -32.67
Df Residuals:                       3   BIC:                            -31.48
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                                     coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
Intercept   

In [ ]:
# Conviction Log Normalized
np.random.seed(29)
formula = "Log_Conviction ~ Boudin + Year + Med_Arrest_To_Close + Med_Arrest_To_Close_Conviction + Med_Arrest_To_Close_Dismissal"
model = smf.gls(formula=formula, data=sf_cr_all)
result = model.fit()
print(result.summary())

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


                            GLS Regression Results                            
Dep. Variable:         Log_Conviction   R-squared:                       0.997
Model:                            GLS   Adj. R-squared:                  0.992
Method:                 Least Squares   F-statistic:                     203.6
Date:                Thu, 21 Sep 2023   Prob (F-statistic):           0.000539
Time:                        01:24:28   Log-Likelihood:                 21.081
No. Observations:                   9   AIC:                            -30.16
Df Residuals:                       3   BIC:                            -28.98
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                                     coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
Intercept   

/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=9
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


## Violent Crimes

### 2023 Inclusive

In [ ]:
# This temporarily disables warnings; A warning of a depreciated opperation displayed
warnings.filterwarnings('ignore')

sf_cr_violent = sf_case_resolutions_violent.copy()
sf_cr_violent["Conviction"] = sf_cr_violent["Conviction"].astype(int)

sf_cr_violent['Log_Conviction'] = np.log(sf_cr_violent['Conviction'])
sf_cr_violent['Log_Total'] = np.log(sf_cr_violent['Total'])
sf_cr_violent['Boudin'] = sf_cr_violent.apply(lambda row : year_marker(row['Year'],
                                                                             2020, 2022),
                                                    axis = 1)
sf_cr_violent['Before_COVID'] = sf_cr_violent.apply(lambda row :
                                                              year_marker(row['Year'],
                                                                          2020, mode = 'before'),
                                                              axis = 1)



sf_cr_violent_proj = sf_cr_violent.copy() #Creates defensive copy
sf_cr_violent_proj.iloc[8, 1] = sf_cr_violent_proj.iloc[9, 1] #Moves 2023* to 2023 slot

# Drops '2023*' Row
sf_cr_violent_proj = sf_cr_violent_proj.drop(9)

# Drops '2023*' Row
sf_cr_violent = sf_cr_violent.drop(9)

# Addresses type issues
sf_cr_violent["Year"] = sf_cr_violent["Year"].astype(int)
sf_cr_violent["Med_Arrest_To_Close"] = sf_cr_violent["Med_Arrest_To_Close"].astype(int)
sf_cr_violent["Med_Arrest_To_Close_Conviction"] = sf_cr_violent["Med_Arrest_To_Close_Conviction"].astype(int)
sf_cr_violent["Med_Arrest_To_Close_Dismissal"] = sf_cr_violent["Med_Arrest_To_Close_Dismissal"].astype(int)

sf_cr_violent_proj["Year"] = sf_cr_violent_proj["Year"].astype(int)
sf_cr_violent_proj["Med_Arrest_To_Close"] = sf_cr_violent_proj["Med_Arrest_To_Close"].astype(int)
sf_cr_violent_proj["Med_Arrest_To_Close_Conviction"] = sf_cr_violent_proj["Med_Arrest_To_Close_Conviction"].astype(int)
sf_cr_violent_proj["Med_Arrest_To_Close_Dismissal"] = sf_cr_violent_proj["Med_Arrest_To_Close_Dismissal"].astype(int)

# Gets rid of 2023 Row
sf_cr_violent_no2023 = sf_cr_violent[sf_cr_violent["Year"] != 2023]

# This resets warnings
warnings.resetwarnings()

In [ ]:
# 2023 Raw & Total Log Normalized
np.random.seed(29)
formula = "Log_Total ~ Boudin + Year + Before_COVID"
model = smf.gls(formula=formula, data=sf_cr_violent)
result = model.fit()
print(result.summary())

                            GLS Regression Results                            
Dep. Variable:              Log_Total   R-squared:                       0.833
Model:                            GLS   Adj. R-squared:                  0.733
Method:                 Least Squares   F-statistic:                     8.307
Date:                Thu, 21 Sep 2023   Prob (F-statistic):             0.0218
Time:                        01:26:44   Log-Likelihood:                 10.557
No. Observations:                   9   AIC:                            -13.11
Df Residuals:                       5   BIC:                            -12.32
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept     -103.5084     58.666     -1.764   

/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=9
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [ ]:
# 2023 Projected & Total Log Normalized
np.random.seed(29)
formula = "Log_Total ~ Boudin + Year + Before_COVID"
model = smf.gls(formula=formula, data=sf_cr_violent_proj)
result = model.fit()
print(result.summary())

                            GLS Regression Results                            
Dep. Variable:              Log_Total   R-squared:                       0.833
Model:                            GLS   Adj. R-squared:                  0.733
Method:                 Least Squares   F-statistic:                     8.307
Date:                Thu, 21 Sep 2023   Prob (F-statistic):             0.0218
Time:                        01:27:09   Log-Likelihood:                 10.557
No. Observations:                   9   AIC:                            -13.11
Df Residuals:                       5   BIC:                            -12.32
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept     -103.5084     58.666     -1.764   

/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=9
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [ ]:
# 2023 Raw & Conviction Log Normalized
np.random.seed(29)
formula = "Log_Conviction ~ Boudin + Year + Before_COVID"
model = smf.gls(formula=formula, data=sf_cr_violent)
result = model.fit()
print(result.summary())

                            GLS Regression Results                            
Dep. Variable:         Log_Conviction   R-squared:                       0.921
Model:                            GLS   Adj. R-squared:                  0.874
Method:                 Least Squares   F-statistic:                     19.48
Date:                Thu, 21 Sep 2023   Prob (F-statistic):            0.00345
Time:                        01:27:36   Log-Likelihood:                 7.4034
No. Observations:                   9   AIC:                            -6.807
Df Residuals:                       5   BIC:                            -6.018
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept       28.8333     83.283      0.346   

/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=9
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [ ]:
# 2023 Projected & Conviction Log Normalized
np.random.seed(29)
formula = "Log_Conviction ~ Boudin + Year + Before_COVID"
model = smf.gls(formula=formula, data=sf_cr_violent_proj)
result = model.fit()
print(result.summary())

                            GLS Regression Results                            
Dep. Variable:         Log_Conviction   R-squared:                       0.921
Model:                            GLS   Adj. R-squared:                  0.874
Method:                 Least Squares   F-statistic:                     19.48
Date:                Thu, 21 Sep 2023   Prob (F-statistic):            0.00345
Time:                        01:27:59   Log-Likelihood:                 7.4034
No. Observations:                   9   AIC:                            -6.807
Df Residuals:                       5   BIC:                            -6.018
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept       28.8333     83.283      0.346   

/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=9
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [ ]:
# 2023 Raw & Successful_Diversion Not Log Normalized
np.random.seed(29)
formula = "Successful_Diversion ~ Boudin + Year + Before_COVID"
model = smf.gls(formula=formula, data=sf_cr_violent)
result = model.fit()
print(result.summary())

                             GLS Regression Results                             
Dep. Variable:     Successful_Diversion   R-squared:                       0.939
Model:                              GLS   Adj. R-squared:                  0.903
Method:                   Least Squares   F-statistic:                     25.79
Date:                  Thu, 21 Sep 2023   Prob (F-statistic):            0.00181
Time:                          01:28:11   Log-Likelihood:                -44.154
No. Observations:                     9   AIC:                             96.31
Df Residuals:                         5   BIC:                             97.10
Df Model:                             3                                         
Covariance Type:              nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept    -1.035e+05   2.

/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=9
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [ ]:
# 2023 Projected & Successful_Diversion Not Log Normalized
np.random.seed(29)
formula = "Successful_Diversion ~ Boudin + Year + Before_COVID"
model = smf.gls(formula=formula, data=sf_cr_violent_proj)
result = model.fit()
print(result.summary())

                             GLS Regression Results                             
Dep. Variable:     Successful_Diversion   R-squared:                       0.939
Model:                              GLS   Adj. R-squared:                  0.903
Method:                   Least Squares   F-statistic:                     25.79
Date:                  Thu, 21 Sep 2023   Prob (F-statistic):            0.00181
Time:                          01:28:49   Log-Likelihood:                -44.154
No. Observations:                     9   AIC:                             96.31
Df Residuals:                         5   BIC:                             97.10
Df Model:                             3                                         
Covariance Type:              nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept    -1.035e+05   2.

/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=9
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [ ]:
# 2023 Raw & Med_Arrest_To_Close Not Log Normalized
np.random.seed(29)
formula = "Med_Arrest_To_Close ~ Boudin + Year + Before_COVID"
model = smf.gls(formula=formula, data=sf_cr_violent)
result = model.fit()
print(result.summary())

                             GLS Regression Results                            
Dep. Variable:     Med_Arrest_To_Close   R-squared:                       0.942
Model:                             GLS   Adj. R-squared:                  0.907
Method:                  Least Squares   F-statistic:                     26.89
Date:                 Thu, 21 Sep 2023   Prob (F-statistic):            0.00164
Time:                         01:29:03   Log-Likelihood:                -43.566
No. Observations:                    9   AIC:                             95.13
Df Residuals:                        5   BIC:                             95.92
Df Model:                            3                                         
Covariance Type:             nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept    -4.225e+04    2.4e+04    

/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=9
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [ ]:
# 2023 Projected & Med_Arrest_To_Close Not Log Normalized
np.random.seed(29)
formula = "Med_Arrest_To_Close ~ Boudin + Year + Before_COVID"
model = smf.gls(formula=formula, data=sf_cr_violent_proj)
result = model.fit()
print(result.summary())

                             GLS Regression Results                            
Dep. Variable:     Med_Arrest_To_Close   R-squared:                       0.942
Model:                             GLS   Adj. R-squared:                  0.907
Method:                  Least Squares   F-statistic:                     26.89
Date:                 Thu, 21 Sep 2023   Prob (F-statistic):            0.00164
Time:                         01:29:19   Log-Likelihood:                -43.566
No. Observations:                    9   AIC:                             95.13
Df Residuals:                        5   BIC:                             95.92
Df Model:                            3                                         
Covariance Type:             nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept    -4.225e+04    2.4e+04    

/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=9
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [ ]:
# 2023 Raw & Med_Arrest_To_Close_Conviction Not Log Normalized
np.random.seed(29)
formula = "Med_Arrest_To_Close_Conviction ~ Boudin + Year + Before_COVID"
model = smf.gls(formula=formula, data=sf_cr_violent)
result = model.fit()
print(result.summary())

                                  GLS Regression Results                                  
Dep. Variable:     Med_Arrest_To_Close_Conviction   R-squared:                       0.953
Model:                                        GLS   Adj. R-squared:                  0.924
Method:                             Least Squares   F-statistic:                     33.60
Date:                            Thu, 21 Sep 2023   Prob (F-statistic):           0.000972
Time:                                    01:29:31   Log-Likelihood:                -41.987
No. Observations:                               9   AIC:                             91.97
Df Residuals:                                   5   BIC:                             92.76
Df Model:                                       3                                         
Covariance Type:                        nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
---------

/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=9
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [ ]:
# 2023 Projected & Med_Arrest_To_Close_Conviction Not Log Normalized
np.random.seed(29)
formula = "Med_Arrest_To_Close_Conviction ~ Boudin + Year + Before_COVID"
model = smf.gls(formula=formula, data=sf_cr_violent_proj)
result = model.fit()
print(result.summary())

                                  GLS Regression Results                                  
Dep. Variable:     Med_Arrest_To_Close_Conviction   R-squared:                       0.953
Model:                                        GLS   Adj. R-squared:                  0.924
Method:                             Least Squares   F-statistic:                     33.60
Date:                            Thu, 21 Sep 2023   Prob (F-statistic):           0.000972
Time:                                    01:29:42   Log-Likelihood:                -41.987
No. Observations:                               9   AIC:                             91.97
Df Residuals:                                   5   BIC:                             92.76
Df Model:                                       3                                         
Covariance Type:                        nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
---------

/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=9
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [ ]:
# 2023 Projected & Med_Arrest_To_Close_Dismissal Not Log Normalized
np.random.seed(29)
formula = "Med_Arrest_To_Close_Dismissal ~ Boudin + Year + Before_COVID"
model = smf.gls(formula=formula, data=sf_cr_violent_proj)
result = model.fit()
print(result.summary())

                                  GLS Regression Results                                 
Dep. Variable:     Med_Arrest_To_Close_Dismissal   R-squared:                       0.909
Model:                                       GLS   Adj. R-squared:                  0.855
Method:                            Least Squares   F-statistic:                     16.74
Date:                           Thu, 21 Sep 2023   Prob (F-statistic):            0.00486
Time:                                   01:29:55   Log-Likelihood:                -43.266
No. Observations:                              9   AIC:                             94.53
Df Residuals:                                  5   BIC:                             95.32
Df Model:                                      3                                         
Covariance Type:                       nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
-------------------

/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=9
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [ ]:
# 2023 Raw & Total Log Normalized
np.random.seed(29)
formula = "Log_Total ~ Boudin + Year + Before_COVID + Med_Arrest_To_Close + Med_Arrest_To_Close_Conviction + Med_Arrest_To_Close_Dismissal"
model = smf.gls(formula=formula, data=sf_cr_violent)
result = model.fit()
print(result.summary())

                            GLS Regression Results                            
Dep. Variable:              Log_Total   R-squared:                       0.967
Model:                            GLS   Adj. R-squared:                  0.867
Method:                 Least Squares   F-statistic:                     9.685
Date:                Thu, 21 Sep 2023   Prob (F-statistic):             0.0965
Time:                        01:30:27   Log-Likelihood:                 17.819
No. Observations:                   9   AIC:                            -21.64
Df Residuals:                       2   BIC:                            -20.26
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                                     coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
Intercept   

/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=9
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [ ]:
# 2023 Projected & Total Log Normalized
np.random.seed(29)
formula = "Log_Total ~ Boudin + Year + Before_COVID + Med_Arrest_To_Close + Med_Arrest_To_Close_Conviction + Med_Arrest_To_Close_Dismissal"
model = smf.gls(formula=formula, data=sf_cr_violent_proj)
result = model.fit()
print(result.summary())

                            GLS Regression Results                            
Dep. Variable:              Log_Total   R-squared:                       0.967
Model:                            GLS   Adj. R-squared:                  0.867
Method:                 Least Squares   F-statistic:                     9.685
Date:                Thu, 21 Sep 2023   Prob (F-statistic):             0.0965
Time:                        01:30:57   Log-Likelihood:                 17.819
No. Observations:                   9   AIC:                            -21.64
Df Residuals:                       2   BIC:                            -20.26
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                                     coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
Intercept   

/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=9
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [ ]:
# 2023 Raw & Conviction Log Normalized
np.random.seed(29)
formula = "Log_Conviction ~ Boudin + Year + Before_COVID + Med_Arrest_To_Close + Med_Arrest_To_Close_Conviction + Med_Arrest_To_Close_Dismissal"
model = smf.gls(formula=formula, data=sf_cr_violent)
result = model.fit()
print(result.summary())

                            GLS Regression Results                            
Dep. Variable:         Log_Conviction   R-squared:                       0.992
Model:                            GLS   Adj. R-squared:                  0.970
Method:                 Least Squares   F-statistic:                     43.61
Date:                Thu, 21 Sep 2023   Prob (F-statistic):             0.0226
Time:                        01:31:10   Log-Likelihood:                 17.937
No. Observations:                   9   AIC:                            -21.87
Df Residuals:                       2   BIC:                            -20.49
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                                     coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
Intercept   

/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=9
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [ ]:
# 2023 Projected & Conviction Log Normalized
np.random.seed(29)
formula = "Log_Conviction ~ Boudin + Year + Before_COVID + Med_Arrest_To_Close + Med_Arrest_To_Close_Conviction + Med_Arrest_To_Close_Dismissal"
model = smf.gls(formula=formula, data=sf_cr_violent_proj)
result = model.fit()
print(result.summary())

                            GLS Regression Results                            
Dep. Variable:         Log_Conviction   R-squared:                       0.992
Model:                            GLS   Adj. R-squared:                  0.970
Method:                 Least Squares   F-statistic:                     43.61
Date:                Thu, 21 Sep 2023   Prob (F-statistic):             0.0226
Time:                        01:31:35   Log-Likelihood:                 17.937
No. Observations:                   9   AIC:                            -21.87
Df Residuals:                       2   BIC:                            -20.49
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                                     coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
Intercept   

/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=9
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


### 2023 Exclusive
(Note: Can't Use Before_COVID because between (2015 & 2022), Boudin & Before_COVID are **Linearly Dependent**)

In [ ]:
# Total Log Normalized
np.random.seed(29)
formula = "Log_Total ~ Boudin + Year"
model = smf.gls(formula=formula, data=sf_cr_violent_no2023)
result = model.fit()
print(result.summary())

                            GLS Regression Results                            
Dep. Variable:              Log_Total   R-squared:                       0.774
Model:                            GLS   Adj. R-squared:                  0.684
Method:                 Least Squares   F-statistic:                     8.564
Date:                Thu, 21 Sep 2023   Prob (F-statistic):             0.0243
Time:                        01:32:16   Log-Likelihood:                 8.9127
No. Observations:                   8   AIC:                            -11.83
Df Residuals:                       5   BIC:                            -11.59
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept   -102.7817     58.492     -1.757      0.1

/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=8
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [ ]:
# Conviction Log Normalized
np.random.seed(29)
formula = "Log_Conviction ~ Boudin + Year"
model = smf.gls(formula=formula, data=sf_cr_violent_no2023)
result = model.fit()
print(result.summary())

                            GLS Regression Results                            
Dep. Variable:         Log_Conviction   R-squared:                       0.886
Model:                            GLS   Adj. R-squared:                  0.841
Method:                 Least Squares   F-statistic:                     19.47
Date:                Thu, 21 Sep 2023   Prob (F-statistic):            0.00437
Time:                        01:32:43   Log-Likelihood:                 6.1097
No. Observations:                   8   AIC:                            -6.219
Df Residuals:                       5   BIC:                            -5.981
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     29.6771     83.035      0.357      0.7

/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=8
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [ ]:
# Successful_Diversion Not Log Normalized
np.random.seed(29)
formula = "Successful_Diversion ~ Boudin + Year"
model = smf.gls(formula=formula, data=sf_cr_violent_no2023)
result = model.fit()
print(result.summary())

                             GLS Regression Results                             
Dep. Variable:     Successful_Diversion   R-squared:                       0.934
Model:                              GLS   Adj. R-squared:                  0.907
Method:                   Least Squares   F-statistic:                     35.18
Date:                  Thu, 21 Sep 2023   Prob (F-statistic):            0.00113
Time:                          01:33:00   Log-Likelihood:                -39.719
No. Observations:                     8   AIC:                             85.44
Df Residuals:                         5   BIC:                             85.68
Df Model:                             2                                         
Covariance Type:              nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept  -1.034e+05   2.55e+04

/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=8
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [ ]:
# Med_Arrest_To_Close Not Log Normalized
np.random.seed(29)
formula = "Med_Arrest_To_Close ~ Boudin + Year"
model = smf.gls(formula=formula, data=sf_cr_violent_no2023)
result = model.fit()
print(result.summary())

                             GLS Regression Results                            
Dep. Variable:     Med_Arrest_To_Close   R-squared:                       0.937
Model:                             GLS   Adj. R-squared:                  0.912
Method:                  Least Squares   F-statistic:                     37.25
Date:                 Thu, 21 Sep 2023   Prob (F-statistic):           0.000992
Time:                         01:33:12   Log-Likelihood:                -39.196
No. Observations:                    8   AIC:                             84.39
Df Residuals:                        5   BIC:                             84.63
Df Model:                            2                                         
Covariance Type:             nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept  -4.233e+04   2.39e+04     -1.76

/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=8
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [ ]:
# Med_Arrest_To_Close_Conviction Not Log Normalized
np.random.seed(29)
formula = "Med_Arrest_To_Close_Conviction ~ Boudin + Year"
model = smf.gls(formula=formula, data=sf_cr_violent_no2023)
result = model.fit()
print(result.summary())

                                  GLS Regression Results                                  
Dep. Variable:     Med_Arrest_To_Close_Conviction   R-squared:                       0.949
Model:                                        GLS   Adj. R-squared:                  0.929
Method:                             Least Squares   F-statistic:                     46.92
Date:                            Thu, 21 Sep 2023   Prob (F-statistic):           0.000575
Time:                                    01:33:38   Log-Likelihood:                -37.793
No. Observations:                               8   AIC:                             81.59
Df Residuals:                                   5   BIC:                             81.82
Df Model:                                       2                                         
Covariance Type:                        nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
-----------

/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=8
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [ ]:
# 2023 Projected & Med_Arrest_To_Close_Dismissal Not Log Normalized
np.random.seed(29)
formula = "Med_Arrest_To_Close_Dismissal ~ Boudin + Year"
model = smf.gls(formula=formula, data=sf_cr_violent_proj)
result = model.fit()
print(result.summary())

                                  GLS Regression Results                                 
Dep. Variable:     Med_Arrest_To_Close_Dismissal   R-squared:                       0.858
Model:                                       GLS   Adj. R-squared:                  0.811
Method:                            Least Squares   F-statistic:                     18.20
Date:                           Thu, 21 Sep 2023   Prob (F-statistic):            0.00284
Time:                                   01:33:54   Log-Likelihood:                -45.275
No. Observations:                              9   AIC:                             96.55
Df Residuals:                                  6   BIC:                             97.14
Df Model:                                      2                                         
Covariance Type:                       nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
---------------------

/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=9
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [ ]:
# Total Log Normalized
np.random.seed(29)
formula = "Log_Total ~ Boudin + Year + Med_Arrest_To_Close + Med_Arrest_To_Close_Conviction + Med_Arrest_To_Close_Dismissal"
model = smf.gls(formula=formula, data=sf_cr_violent)
result = model.fit()
print(result.summary())

                            GLS Regression Results                            
Dep. Variable:              Log_Total   R-squared:                       0.967
Model:                            GLS   Adj. R-squared:                  0.911
Method:                 Least Squares   F-statistic:                     17.41
Date:                Thu, 21 Sep 2023   Prob (F-statistic):             0.0200
Time:                        01:34:18   Log-Likelihood:                 17.813
No. Observations:                   9   AIC:                            -23.63
Df Residuals:                       3   BIC:                            -22.44
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                                     coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
Intercept   

/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=9
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [ ]:
# Conviction Log Normalized
np.random.seed(29)
formula = "Log_Conviction ~ Boudin + Year + Med_Arrest_To_Close + Med_Arrest_To_Close_Conviction + Med_Arrest_To_Close_Dismissal"
model = smf.gls(formula=formula, data=sf_cr_violent)
result = model.fit()
print(result.summary())

                            GLS Regression Results                            
Dep. Variable:         Log_Conviction   R-squared:                       0.989
Model:                            GLS   Adj. R-squared:                  0.971
Method:                 Least Squares   F-statistic:                     54.66
Date:                Thu, 21 Sep 2023   Prob (F-statistic):            0.00380
Time:                        01:34:51   Log-Likelihood:                 16.323
No. Observations:                   9   AIC:                            -20.65
Df Residuals:                       3   BIC:                            -19.46
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                                     coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
Intercept   

/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=9
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


# This is a test to see if anything  changed